# E-OBS: quickstart

Download E-OBS daily mean temperature for Europe (ensemble mean, 0.1 deg,
full 1950-present period), subset to the UK, and plot a climatology map.

See [`docs/e-obs/README.md`](../docs/e-obs/README.md) for the full
reference.

Note: E-OBS returns the whole 1950-present history in a single file. The
file is ~1 GB per variable; the first download may take a while.

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
VARIABLE = "mean_temperature"
PRODUCT_TYPE = "ensemble_mean"
GRID_RESOLUTION = "0.1deg"
VERSION = "29.0e"
PERIOD = "full_period"
OUTPUT_DIR = "../data/e-obs"
OUTPUT_FILENAME = "e_obs_quickstart.nc"
# ==================================================================

## Imports and environment check

In [ ]:
import sys
from importlib.metadata import version as _v
from pathlib import Path

import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print(f"Python       {sys.version.split()[0]}")
for pkg in ["cdsapi", "xarray", "matplotlib", "cartopy", "netcdf4"]:
    print(f"{pkg:12} {_v(pkg)}")

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "CLAUDE.md").exists():
            return parent
    raise RuntimeError("Could not find repo root.")

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from common.credentials import check_cds_credentials
check_cds_credentials()
print("\nCDS credentials found.")

## Download the whole-period file

E-OBS is served as a single NetCDF per variable per version covering the
full 1950-present archive. Subsetting by time or region happens
client-side after download.

In [ ]:
from scripts.e_obs_download import download  # noqa: E402

output_path = download(
    variable=VARIABLE,
    product_type=PRODUCT_TYPE,
    grid_resolution=GRID_RESOLUTION,
    version=VERSION,
    period=PERIOD,
    output_dir=OUTPUT_DIR,
    output_filename=OUTPUT_FILENAME,
)
print(f"Saved: {output_path} ({output_path.stat().st_size / 1e9:.2f} GB)")

## Open and subset

E-OBS uses `latitude` and `longitude` (or sometimes `lat`/`lon`) and a
`time` dimension. Land-only means ocean cells are masked (NaN). Subset
to the UK bbox and a single decade for the quickstart.

In [ ]:
ds = xr.open_dataset(output_path)
print(ds)

# E-OBS NetCDF variable names follow ECA&D short codes.
# Map from CDS API name -> expected NetCDF short name.
_EOBS_VAR_MAP = {
    "mean_temperature": "tg",
    "maximum_temperature": "tx",
    "minimum_temperature": "tn",
    "precipitation_amount": "rr",
    "mean_sea_level_pressure": "pp",
    "relative_humidity": "hu",
    "global_radiation": "qq",
    "wind_speed": "fg",
}
expected = _EOBS_VAR_MAP.get(VARIABLE)
if expected and expected in ds.data_vars:
    var = expected
else:
    candidates = [v for v in ds.data_vars if v.lower() in _EOBS_VAR_MAP.values()]
    if not candidates:
        raise KeyError(
            f"No known E-OBS variable found in {output_path}. "
            f"Expected one of {list(_EOBS_VAR_MAP.values())}, got {list(ds.data_vars)}."
        )
    var = candidates[0]
print(f"\nMain variable: {var}")

# E-OBS latitude is ascending, but sortby makes the slice robust regardless.
ds = ds.sortby("latitude").sortby("longitude")

uk = ds[var].sel(
    latitude=slice(49, 61),
    longitude=slice(-8, 2),
    time=slice("2015-01-01", "2024-12-31"),
)
print(f"UK subset shape: {uk.shape}")

## Plot the decadal mean

Averaging over 2015-2024 gives a smooth climatological field useful as
a reference baseline.

In [ ]:
climatology = uk.mean("time")

fig = plt.figure(figsize=(10, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
climatology.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    cbar_kwargs={"label": f"Decadal mean {var} (C)"},
)
ax.coastlines(resolution="50m", linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor="gray")
gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = False
gl.right_labels = False
ax.set_title(f"E-OBS {var} decadal mean, UK, 2015-2024 (v{VERSION})")
plt.tight_layout()
plt.show()

## Next steps

- **Full reference**: [`docs/e-obs/README.md`](../docs/e-obs/README.md)
- **Compare against ERA5**: pull ERA5 `2m_temperature` for the same
  region and decade, and plot the difference. The reanalysis and the
  observation-derived grid should agree to within 1-2 C at most
  locations.
- **Uncertainty**: set `PRODUCT_TYPE = "ensemble_members"` to pull the
  100 realisations; compute a per-cell spread for uncertainty maps.
- **Different variable**: try `precipitation_amount` for daily rainfall.
- **HadUK-Grid** for UK-specific work: the Met Office alternative uses
  a denser UK station network and is at 1 km resolution.